# 15 · Does part 1 still hold?

Part 1 (3,469 postings, scraped in October 2026 from ITviec, TopCV, VietnamWorks, TopDev and LinkedIn and classified by an LLM from the full description) made three headline claims: **data/AI/ML is the biggest group but one of the least open to newcomers**; **only about a quarter of IT postings are open to interns, freshers and juniors** (backend and DevOps under 15 %); and **remote is about 3 %**. This notebook recomputes those numbers from the part 1 dataset itself and sets them beside the Techmap feed, for January to October 2026, all sources and LinkedIn alone (the one site both datasets share).

Differences to keep in mind: the two datasets cover different sites, part 1 read the level, work mode and role from the *full description* with an LLM, while this project reads them from the title, the board's labels and the stated experience, and part 1's tech filter excluded product, BA and project roles by design.

In [1]:
from vnjobs.analysis import core as a
from vnjobs.analysis import reconciliation as r

a.apply_style()
con = a.connect()
df = a.load_features(con)

In [2]:
part1 = r.load_part1()
assert part1 is not None, "part 1 dataset not found next to this repo"
scopes = {
    "Techmap 2026, all sources": r.part2_window(df),
    "Techmap 2026, LinkedIn": r.part2_window(df, "linkedin_vn"),
}
everything = scopes["Techmap 2026, all sources"]
linkedin = scopes["Techmap 2026, LinkedIn"]
print(f"part 1: {len(part1):,}; Techmap: {len(everything):,} (all), {len(linkedin):,} (LinkedIn)")
part1.site.value_counts().rename_axis("site").reset_index(name="postings")

part 1: 3,469; Techmap: 36,477 (all), 22,845 (LinkedIn)


,site,postings
0,topcv,1323
1,linkedin,625
2,vietnamworks,591
3,itviec,555
4,topdev,375


## 1. Is data/AI/ML still the biggest group?

Share of postings in each of part 1's categories.

In [3]:
mix = r.role_mix(part1, scopes)
a.save_table(mix, "15-role-mix")
mix

,category,part 1,"Techmap 2026, all sources","Techmap 2026, LinkedIn"
0,Data / AI / ML,18.0,20.9,24.3
1,Other,17.4,29.3,22.2
2,Infrastructure / IT Support,14.3,7.3,6.0
3,Backend,10.9,8.6,11.1
4,QA / Testing,9.2,12.8,12.3
5,Fullstack,7.9,5.0,6.5
6,DevOps / SRE / Cloud,5.8,7.8,8.5
7,Product / BA / Agile,5.5,0.3,0.3
8,Security / Cyber,5.0,3.5,3.5
9,Game Development,2.8,0.7,0.9


## 2. How open is IT to newcomers?

Entry-level share (intern, fresher, junior) of postings with a known level, per category. Techmap appears twice: with this project's full definition, and counting only levels read from the title or the board's label (closer to how part 1 read it).

In [4]:
entry = r.entry_by_category(part1, scopes)
a.save_table(entry, "15-entry-level")
entry

,category,part 1,"Techmap 2026, all sources, full","Techmap 2026, all sources, title/label only","Techmap 2026, LinkedIn, full","Techmap 2026, LinkedIn, title/label only"
0,Backend,14.2,29.1,29.2,30.1,30.4
1,Data / AI / ML,18.9,32.1,32.0,33.1,33.2
2,DevOps / SRE / Cloud,12.7,24.8,25.0,26.5,26.7
3,Frontend,15.6,28.6,29.2,28.9,29.4
4,Fullstack,17.0,27.1,27.4,27.6,28.3
5,Game Development,54.8,45.8,47.6,53.1,53.9
6,Infrastructure / IT Support,38.7,46.1,47.2,48.0,48.8
7,Mobile,12.3,30.5,31.6,36.2,36.5
8,Other,41.0,35.4,33.3,35.7,36.1
9,Product / BA / Agile,22.6,27.1,31.9,33.8,34.4


In [5]:
by_site = part1.assign(known=part1.seniority_category != "Unknown")
by_site = (
    by_site[by_site.known]
    .groupby("site")
    .seniority_category.agg(
        postings="size", entry_pct=lambda s: round(100 * r.is_entry_part1(s).mean(), 1)
    )
)
a.save_table(by_site.reset_index(), "15-part1-entry-by-site")
by_site

,postings,entry_pct
site,,
itviec,527,14.6
linkedin,564,20.4
topcv,1323,38.6
topdev,359,39.0
vietnamworks,588,8.5


## 3. Is remote about 3 %?

Remote, hybrid and on-site as a share of **all** postings and of the postings that **state** a mode. Part 1 read a mode for most postings from the description; the Techmap feed states one for about a quarter.

In [6]:
modes = r.work_mode(part1, scopes)
a.save_table(modes, "15-work-mode")
modes

,dataset,mode,pct_of_all_postings,pct_of_stated,pct_stating_a_mode
0,part 1,remote,3.1,4.0,78.0
1,part 1,hybrid,7.7,9.8,78.0
2,part 1,onsite,67.2,86.2,78.0
3,"Techmap 2026, all sources",remote,4.0,15.8,25.3
4,"Techmap 2026, all sources",hybrid,6.8,26.7,25.3
5,"Techmap 2026, all sources",onsite,14.6,57.4,25.3
6,"Techmap 2026, LinkedIn",remote,5.2,29.1,17.8
7,"Techmap 2026, LinkedIn",hybrid,7.8,44.1,17.8
8,"Techmap 2026, LinkedIn",onsite,4.8,26.8,17.8


## 4. Cities and technologies

City split of postings with a known city, and each dataset's ten most-named technologies (the two dictionaries differ, so compare the ranking, not the percentages).

In [7]:
cities = r.locations(part1, scopes)
a.save_table(cities, "15-cities")
cities

,city,part 1,"Techmap 2026, all sources","Techmap 2026, LinkedIn"
0,Hà Nội,47.4,30.0,32.5
1,Hồ Chí Minh,35.2,44.3,44.2
2,Other,15.6,24.2,22.1
3,Đà Nẵng,1.8,1.5,1.2


In [8]:
tech = r.top_technologies(part1, scopes["Techmap 2026, all sources"])
a.save_table(tech, "15-top-technologies")
tech

,rank,part 1,Techmap
0,1,Python (33 %),Python (42 %)
1,2,SQL (20 %),CI/CD (31 %)
2,3,Docker (17 %),AWS (28 %)
3,4,Java (17 %),SQL (27 %)
4,5,PostgreSQL (15 %),Git (24 %)
5,6,Kubernetes (14 %),Azure (21 %)
6,7,MySQL (11 %),Docker (21 %)
7,8,JavaScript (11 %),Java (20 %)
8,9,TypeScript (9 %),Kubernetes (18 %)
9,10,Git (9 %),REST API (16 %)


## Findings

**What holds**

1. **Data/AI/ML is still the biggest named group.** 18.0 % of postings in part 1, 20.9 % in Techmap for 2026 (24.3 % on LinkedIn). In Techmap the "Other" bucket is larger (29.3 %) but it holds the technicians and non-software engineers found in `03-roles`, and Product/BA/Agile is nearly absent (0.3 % against 5.5 %) because part 1's tech filter excluded those roles by design.
2. **Remote is a few percent of all IT postings.** Part 1: 3.1 %. Techmap 2026: 4.0 % (5.2 % on LinkedIn).
3. **Python is first in both.** 33 % of part 1's postings that name a technology, 42 % in Techmap.

**What changes**

4. **"About a quarter are open to newcomers" is closer to a third.** Entry-level share of postings with a known level: part 1 26.6 %, Techmap 33.4 % (34.0 % on LinkedIn). It also depends on the site: inside part 1 it ranges from 8.5 % (VietnamWorks) and 14.6 % (ITviec) to 38.6 % (TopCV) and 39.0 % (TopDev). On LinkedIn, which both datasets include, part 1 gets 20.4 % and Techmap 34.0 %, so the same site gives different answers when the level is read from the full description by an LLM (part 1) or from the title, the board's label and the stated experience (here).
5. **The ranking of groups holds, the sizes do not.** In both datasets game (55 % and 46–53 %) and IT support (39 % and 46–48 %) are the most open; backend, DevOps, mobile and frontend are among the least open. But part 1's "backend and DevOps under 15 %" becomes 29.1 % and 24.8 % in Techmap (30.1 % and 26.5 % on LinkedIn), and part 1's "Data/AI/ML is one of the least open groups" (18.9 %) becomes 32.1 % (33.1 % on LinkedIn), roughly the average for all IT (33.4 %).
6. **"Remote is only 3 %" is true of all postings but not of postings that state a mode.** Part 1 read a mode for 78 % of postings from the description; the Techmap feed states one for 25 %. Remote is 4.0 % of the stated modes in part 1 and 15.8 % (29.1 % on LinkedIn) in Techmap. The honest statement is "remote is about 3–5 % of all IT postings".
7. **Cities and technologies depend on the sites.** Part 1 has Hà Nội ahead (47 % against 35 % for Hồ Chí Minh); Techmap 2026 has Hồ Chí Minh ahead (44 % against 30 %), also on LinkedIn alone (44 % against 33 %). "Docker in third place" does not hold: Docker is seventh in Techmap (21 %), behind CI/CD, AWS, SQL, Git and Azure; SQL is fourth.

**Suggested corrections for the video** (if it repeats part 1's claims): say "roughly a quarter to a third" for entry-level, qualify "backend and DevOps under 15 %" and "Data/AI/ML among the least open" as true of part 1's sample only, and say remote is "about 3–5 % of all postings", not "about 3 % of those that say".

**What this notebook cannot say:** which dataset is closer to the truth. They cover different sites and read the fields differently.
